<a href="https://colab.research.google.com/github/hissain/ml/blob/main/codes/advance/Copy_of_SpeculativeDecoding.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import time
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
TARGET_MODEL = "facebook/opt-1.3b"   # Large model (verifier)
DRAFT_MODEL  = "facebook/opt-125m"   # Small model (drafter)

print(f"Loading tokenizer from {TARGET_MODEL} ...")
tokenizer = AutoTokenizer.from_pretrained(TARGET_MODEL)

print(f"Loading target model ({TARGET_MODEL}) ...")
target_model = AutoModelForCausalLM.from_pretrained(
    TARGET_MODEL,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto",
)

print(f"Loading draft model ({DRAFT_MODEL}) ...")
draft_model = AutoModelForCausalLM.from_pretrained(
    DRAFT_MODEL,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto",
)

target_model.eval()
draft_model.eval()
print("\n Both models loaded successfully!")

Loading tokenizer from facebook/opt-1.3b ...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/653 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/685 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/651 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/251M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]


 Both models loaded successfully!


In [ ]:
def benchmark_standard(prompt: str, max_new_tokens: int = 200, runs: int = 3):
    """Standard autoregressive decoding with the large model only."""
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    times = []

    for i in range(runs):
        torch.cuda.synchronize() if device == "cuda" else None
        t0 = time.perf_counter()

        with torch.no_grad():
            output_ids = target_model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,          # greedy for fair comparison
                temperature=1.0,
            )

        torch.cuda.synchronize() if device == "cuda" else None
        t1 = time.perf_counter()
        times.append(t1 - t0)
        print(f"  Run {i+1}/{runs}: {t1-t0:.2f}s")

    output_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    tokens_generated = output_ids.shape[1] - inputs["input_ids"].shape[1]
    avg_time = sum(times) / len(times)
    return output_text, avg_time, tokens_generated

In [ ]:
def benchmark_speculative(prompt: str, max_new_tokens: int = 200, runs: int = 3,
                          num_assistant_tokens: int = 5):
    """
    Speculative decoding via HuggingFace's `assistant_model` parameter.
    The draft model proposes `num_assistant_tokens` tokens per step;
    the target model verifies them in one forward pass.
    """
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    times = []

    for i in range(runs):
        torch.cuda.synchronize() if device == "cuda" else None
        t0 = time.perf_counter()

        with torch.no_grad():
            output_ids = target_model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                temperature=1.0,
                assistant_model=draft_model,          # ← enables speculative decoding
                num_assistant_tokens=num_assistant_tokens,
            )

        torch.cuda.synchronize() if device == "cuda" else None
        t1 = time.perf_counter()
        times.append(t1 - t0)
        print(f"  Run {i+1}/{runs}: {t1-t0:.2f}s")

    output_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    tokens_generated = output_ids.shape[1] - inputs["input_ids"].shape[1]
    avg_time = sum(times) / len(times)
    return output_text, avg_time, tokens_generated

In [ ]:
def print_results(label, output_text, avg_time, tokens_generated):
    tok_per_sec = tokens_generated / avg_time
    print(f"\n{'='*60}")
    print(f"  {label}")
    print(f"{'='*60}")
    print(f"Avg time        : {avg_time:.2f}s")
    print(f"Tokens generated: {tokens_generated}")
    print(f"Throughput       : {tok_per_sec:.1f} tokens/sec")
    print(f"Output (first 300 chars):\n")
    print(f"     {output_text[:300]}...")
    print(f"{'='*60}")
    return tok_per_sec

In [ ]:
PROMPT = (
    "Explain the theory of general relativity in simple terms, "
    "covering spacetime curvature, gravity, and the equivalence principle."
)
MAX_NEW_TOKENS   = 200
RUNS             = 3    # average over N runs for stability
DRAFT_TOKENS     = 5    # tokens the draft model proposes per step

print(f"Prompt: {PROMPT}")
print(f"Max new tokens: {MAX_NEW_TOKENS} | Runs: {RUNS}\n")

# ── Standard decoding ──
print("Running STANDARD decoding ...")
std_text, std_time, std_tokens = benchmark_standard(PROMPT, MAX_NEW_TOKENS, RUNS)
std_tps = print_results("STANDARD DECODING (large model only)", std_text, std_time, std_tokens)

# ── Speculative decoding ──
print("\nRunning SPECULATIVE decoding ...")
spec_text, spec_time, spec_tokens = benchmark_speculative(PROMPT, MAX_NEW_TOKENS, RUNS, DRAFT_TOKENS)
spec_tps = print_results("SPECULATIVE DECODING (draft + target)", spec_text, spec_time, spec_tokens)

Prompt: Explain the theory of general relativity in simple terms, covering spacetime curvature, gravity, and the equivalence principle.
Max new tokens: 200 | Runs: 3

Running STANDARD decoding ...
  Run 1/3: 12.91s
  Run 2/3: 3.28s


[transformers] Passing `generation_config` together with generation-related arguments=({'min_new_tokens', 'use_cache', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


  Run 3/3: 3.28s

  STANDARD DECODING (large model only)
Avg time        : 6.49s
Tokens generated: 200
Throughput       : 30.8 tokens/sec
Output (first 300 chars):

     Explain the theory of general relativity in simple terms, covering spacetime curvature, gravity, and the equivalence principle.

Theory of General Relativity

General relativity is a theory of the universe that describes the behavior of objects in space and time. It is a theory that describes the be...

Running SPECULATIVE decoding ...
  Run 1/3: 2.65s
  Run 2/3: 2.36s
  Run 3/3: 2.30s

  SPECULATIVE DECODING (draft + target)
Avg time        : 2.44s
Tokens generated: 200
Throughput       : 82.0 tokens/sec
Output (first 300 chars):

     Explain the theory of general relativity in simple terms, covering spacetime curvature, gravity, and the equivalence principle.

Theory of General Relativity

General relativity is a theory of the universe that describes the behavior of objects in space and time. It is a theory that des

In [ ]:
speedup = std_time / spec_time
outputs_match = std_text.strip() == spec_text.strip()

print("\n" + "#" * 60)
print("#" + " " * 20 + "BENCHMARK SUMMARY" + " " * 21 + "#")
print("#" * 60)
print(f"  Standard decoding  : {std_time:.2f}s  ({std_tps:.1f} tok/s)")
print(f"  Speculative decoding: {spec_time:.2f}s  ({spec_tps:.1f} tok/s)")
print(f"  Speedup             : {speedup:.2f}x")
print(f"  Outputs identical   : {'Yes' if outputs_match else 'Minor diff (expected with sampling)'}")
print("#" * 60)
print()
print("How speculative decoding works:")
print(f"  1. Draft model (opt-125m) proposes {DRAFT_TOKENS} tokens cheaply.")
print("  2. Target model (opt-1.3b) verifies all drafts in ONE forward pass.")
print("  3. Accepted tokens are kept; the first rejected token is resampled.")
print("  4. Net result: fewer target-model forward passes → faster generation.")


############################################################
#                    BENCHMARK SUMMARY                     #
############################################################
  Standard decoding  : 6.49s  (30.8 tok/s)
  Speculative decoding: 2.44s  (82.0 tok/s)
  Speedup             : 2.66x
  Outputs identical   : Yes
############################################################

How speculative decoding works:
  1. Draft model (opt-125m) proposes 5 tokens cheaply.
  2. Target model (opt-1.3b) verifies all drafts in ONE forward pass.
  3. Accepted tokens are kept; the first rejected token is resampled.
  4. Net result: fewer target-model forward passes → faster generation.
